## OLCI Sargassum Detection
`Read scenes from S3 bucket, make daily mean SGI zarr stores`

### 1. Import Libraries

In [1]:
import os
import re
from datetime import datetime
import time

import numpy as np
import xarray as xr
import zarr
import s3fs
from botocore.exceptions import ClientError
from scipy.ndimage import binary_dilation
from pyresample import geometry, kd_tree

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

### 2. Connect to the S3 bucket

In [2]:
# s3fs connection to CloudFerro-backed bucket holding OLCI
fs = s3fs.S3FileSystem(
    client_kwargs = {
        'endpoint_url' : 'https://s3.waw3-1.cloudferro.com',
        'region_name'  : 'waw3-1'},
    config_kwargs = {'signature_version': 's3v4'},
    profile       = 'laurenb')

S3_BUCKET = 'olci_l1_fr' 

### 3. Define the target grid and processing parameters

In [3]:
# shared 300m-equivalent TNA grid every scene gets regridded onto
GRID_RES_DEG = 300 / 111320  # 300m in degrees, at these latitudes
lon_1d = np.arange(-70, -20, GRID_RES_DEG)
lat_1d = np.arange(-5,   20, GRID_RES_DEG)

target_lons, target_lats = np.meshgrid(lon_1d, lat_1d)
target_def = geometry.GridDefinition(lons = target_lons, lats = target_lats)

# processing parameters
buffer_pixels   = 20
cloud_flag_vars = ('saturated', 'partially_saturated', 'invalid')
land_flag_vars  = ('land', 'coastline', 'fresh_inland_water', 'tidal_region')
radbands        = ('Oa08_radiance', 'Oa09_radiance', 'Oa12_radiance', 'Oa16_radiance', 'Oa17_radiance')
radius          = 500  # matched to 300m cells

zarr_root = '/home/nandii/olci_sg/day_zarrs'
os.makedirs(zarr_root, exist_ok = True)

### 4. List scenes in the bucket and group them by day

In [4]:
# bucket each scene into its calendar day, using the start-sensing timestamp in SEN3 folder name:
years_present  = range(2016, 2027)
months_present = range(5, 13)

scene_names = sorted(set(key.split('/')[1] for key in fs.find(f'{S3_BUCKET}/', refresh = True)))

scenes_by_day = {}
for scene_name in scene_names:
    match = re.search(r'_(\d{8}T\d{6})_', scene_name)
    if match is None:
        continue  # not a scene folder (e.g. sgi_zarrs)
    scene_datetime = datetime.strptime(match.group(1), '%Y%m%dT%H%M%S')

    if scene_datetime.year not in years_present or scene_datetime.month not in months_present:
        continue

    scenes_by_day.setdefault(scene_datetime.date(), []).append(scene_name)

print(f"Found {len(scene_names)} scenes across {len(scenes_by_day)} days")

Found 5501 scenes across 1345 days


### 5. Build and save each day's mean SGI to zarr

Builds only the days that are new or missing.
Each scene is regridded and added to a running sum and count, so only one scene is held in memory at a time.

In [ ]:
# main build loop: one pass per day, one inner pass per scene in that day
for day, names in sorted(scenes_by_day.items()):

    zarr_path = os.path.join(zarr_root, f'SGI_{day:%Y%m%d}.zarr')
    if os.path.exists(zarr_path):
        continue

    start_time = time.time()
    print(f"Building {day:%d %B %Y} daily mean: {len(names)} scenes")

    # running sum of SGI and number of valid scenes per grid cell
    sgi_sum = np.zeros(target_lons.shape, dtype = np.float64)
    n_obs   = np.zeros(target_lons.shape, dtype = np.uint16)
    scene_names_used = []

    for scene_name in names:
        # stream this scene's raw files from S3 and compute its SGI field
        try:
            prefix = f'{S3_BUCKET}/{scene_name}'

            with fs.open(f'{prefix}/geo_coordinates.nc', cache_type = 'blockcache') as f:
                geo = xr.open_dataset(f)
                lat = geo['latitude'].values.copy()
                lon = geo['longitude'].values.copy()

            radiances = {}
            for band in radbands:
                with fs.open(f'{prefix}/{band}.nc', cache_type = 'blockcache') as f:
                    ds = xr.open_dataset(f)
                    radiances[band] = ds[band].values.copy()

            band09 = radiances["Oa09_radiance"]
            band12 = radiances["Oa12_radiance"]
            band16 = radiances["Oa16_radiance"]

            with fs.open(f'{prefix}/qualityFlags.nc', cache_type = 'blockcache') as f:
                FLAG_file = xr.open_dataset(f)
                flag_names = FLAG_file['quality_flags'].flag_meanings.split(' ')
                flag_vals  = FLAG_file['quality_flags'].flag_masks
                FLAGS      = FLAG_file['quality_flags'].values.copy()

            # land mask: bitwise OR of the requested flag bits, tested against FLAGS
            land_flag_bits = np.uint64()
            for flag in land_flag_vars:
                try:
                    land_flag_bits = land_flag_bits | flag_vals[flag_names.index(flag)]
                except ValueError:
                    print(flag + " not present")
            land_mask_raw = (FLAGS & land_flag_bits) > 0

            structure = np.ones((3, 3))
            land_mask_buffered = binary_dilation(land_mask_raw, structure = structure, iterations = buffer_pixels)

            # cloud mask: same bitwise approach, different flag set
            cloud_flag_bits = np.uint64()
            for flag in cloud_flag_vars:
                try:
                    cloud_flag_bits = cloud_flag_bits | flag_vals[flag_names.index(flag)]
                except ValueError:
                    print(flag + " not present")
            cloud_mask_raw = (FLAGS & cloud_flag_bits) > 0

            flag_mask_combined = land_mask_buffered | cloud_mask_raw
            flag_mask = flag_mask_combined.astype(float)
            flag_mask[flag_mask == 0.0] = np.nan

            for band_arr in [band09, band12, band16]:
                band_arr[np.isfinite(flag_mask)] = np.nan

            # continuous, radiance-based SGI, NO THRESHOLD
            SGI = (band12 - band09) + (band16 - band09)

        except (FileNotFoundError, ClientError) as err:
            print(f"Skipping {scene_name}: {err}")
            continue

        # regrid this scene's SGI field onto the shared target grid
        swath_def = geometry.SwathDefinition(lons = lon, lats = lat)
        sgi_regridded = kd_tree.resample_nearest(
            swath_def, SGI.astype(float), target_def,
            radius_of_influence = radius, fill_value = np.nan
        )

        # add this scene into the running sum and count, in place
        valid = np.isfinite(sgi_regridded)
        np.add(sgi_sum, sgi_regridded, out = sgi_sum, where = valid)
        n_obs += valid

        scene_names_used.append(scene_name.removesuffix('.SEN3'))
        del sgi_regridded, valid

    if not scene_names_used:
        print(f"No usable scenes for {day:%d %B %Y} \u2014 skipped")
        continue

    # daily mean, NaN where no scene had a valid pixel
    sgi_mean = np.full(target_lons.shape, np.nan)
    np.divide(sgi_sum, n_obs, out = sgi_mean, where = n_obs > 0)
    del sgi_sum

    ds_day = xr.Dataset(
        {'SGI_mean' : (('lat', 'lon'), sgi_mean),
         'n_obs'    : (('lat', 'lon'), n_obs)},
        coords = {
            'lat' : lat_1d,
            'lon' : lon_1d},
        attrs = {
            'title'         : f'OLCI Sargassum SGI daily mean \u2014 {day:%d %B %Y}',
            'date'          : f'{day:%Y-%m-%d}',
            'buffer_pixels' : buffer_pixels,
            'cloud_flags'   : ','.join(cloud_flag_vars),
            'scene_names'   : ','.join(scene_names_used),
            'n_scenes'      : len(scene_names_used)})
    ds_day = ds_day.chunk({'lat': 200, 'lon': 200})

    ds_day.to_zarr(zarr_path, mode = 'w')

    elapsed = time.time() - start_time
    print(f"Wrote {zarr_path} in {elapsed/60:.1f} min ({len(scene_names_used)} scenes)")
    del sgi_mean, n_obs, ds_day

Building 01 May 2016 daily mean: 2 scenes
Wrote /home/nandii/olci_sg/day_zarrs/SGI_20160501.zarr in 1.1 min (2 scenes)
Building 02 May 2016 daily mean: 2 scenes
